<a href="https://colab.research.google.com/github/ksuplee/AI_Agent/blob/main/08_1_Intent_Slot_DialogState.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 실습 08-1: Intent·Slot·Dialog State 개념  

- 기존의 메모리 에이전트에 아래 3가지 요소를 로직으로 주입합니다.  
- Intent(의도): 사용자가 무엇을 원하는지 파악 (예: 소개, 학습_상담, 일상_대화)  
- Slot(슬롯): 의도 수행을 위해 필요한 필수 정보 (예: 이름, 거주지, 학습_주제)  
- Dialog State(대화 상태): 어떤 슬롯이 채워졌고 현재 어느 단계인지 추적  

### 1. 환경 준비
필요한 라이브러리를 설치합니다.

In [ ]:
# 기존 설치를 무시하고 최신 버전으로 강제 재설치합니다.
!pip install -q -U --force-reinstall langchain langchain-community langchain-huggingface langchain-core langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 490.2/490.2 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.4/719.4 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

### 2. Gemini LLM 로드

Google Gemini 모델을 사용하여 더 나은 답변을 시도할 수 있습니다. Gemini 모델을 사용하려면 `google-generativeai` 라이브러리를 설치하고 API 키를 설정해야 합니다.

Gemini API를 사용하려면 API 키가 필요합니다.

아직 키가 없다면 Google AI Studio에서 키를 생성하세요.

1. Google AI Studio 접속
먼저 공식 사이트(https://aistudio.google.com)에 접속합니다. 사용 중인 구글 계정으로 로그인해 주세요.

2. 서비스 약관 동의
처음 접속하신 경우, 생성형 AI 사용을 위한 서비스 약관 동의 팝업이 뜹니다. 내용을 확인하신 후 'Accept' 또는 'Continue' 버튼을 클릭하여 메인 대시보드로 진입합니다.

3. API 키 메뉴 이동  
    - [대시보드] 왼쪽 상단 메뉴 바에서 [Get API key] 항목을 클릭합니다.

4. API 키 생성: 화면 중앙에 보이는 버튼 중 하나를 선택합니다.  

    - [Create API key] in new project: 새로운 프로젝트를 생성하면서 키를 발급받습니다. (처음 만드시는 분들께 권장)

    - Create API key in existing project: 기존에 사용하던 Google Cloud 프로젝트가 있다면 해당 프로젝트를 선택하여 키를 생성합니다.

5. 키 복사 및 안전한 보관:  
팝업창에 생성된 **긴 문자열(API Key)**이 나타납니다. 'Copy' 버튼을 눌러 복사한 뒤, 메모장이나 환경 변수 설정 등 안전한 곳에 저장해 두세요.

    ⚠️ 주의: API 키는 비밀번호와 같습니다. GitHub 같은 공개 저장소에 코드를 올릴 때 키가 노출되지 않도록 주의하세요!

6. 팁: 요금 및 제한 사항 (무료 티어 기준)  

    - Gemini 1.5 Flash: 속도가 빠르고 무료 사용량이 넉넉하여 테스트용으로 좋습니다.  
    - Gemini 1.5 Pro: 복잡한 추론에 적합하지만, 무료 티어에서는 분당 요청 횟수(RPM) 제한이 더 타이트합니다.  
    - 개인정보: 무료 등급 사용 시 입력한 데이터는 모델 학습에 사용될 수 있으므로 민감한 정보는 입력하지 않는 것이 좋습니다.  

Colab에서는 왼쪽 패널의 "🔑" 아래에 키를 `GOOGLE_API_KEY`라는 이름으로 Secrets Manager에 추가하세요. 그런 다음 키를 SDK에 전달합니다.

In [ ]:
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

print("Gemini API 설정 완료")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✅ Gemini API 설정 완료


3. LCEL 기반 지능형 대화 관리

- 기존 history_db에 더해, **슬롯(Slot)**을 관리하는 dialog_state 객체를 추가하여 에이전트가 사용자의 정보를 체크리스트처럼 관리하도록 수정합니다.  

In [ ]:
import google.generativeai as genai
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_community.chat_message_histories import ChatMessageHistory

# 1. 상태 관리 객체 정의 (08-1 개념 반영)
history_db = ChatMessageHistory()
# 에이전트가 채워야 할 체크리스트(Slots)
dialog_state = {
    "intent": None,
    "slots": {"name": None, "location": None, "topic": None},
    "status": "INIT"
}

# 2. Gemini LLM 설정
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
llm = ChatGoogleGenerativeAI(model='gemini-flash-latest', api_key=GOOGLE_API_KEY)

# 3. 08-1 전용 프롬프트 정의 (Intent & Slot 강조)
template = """너는 사용자의 의도(Intent)를 파악하고 필요한 정보(Slot)를 수집하는 전문 상담가야.

[현재 수집된 정보(Dialog State)]
- 이름: {name}
- 거주지: {location}
- 학습주제: {topic}

[대화 기록]
{history}

질문: {question}

지시사항:
1. 사용자가 정보를 제공하면 해당 정보를 기억해.
2. 아직 채워지지 않은 정보(None)가 있다면 자연스럽게 대화하며 물어봐(Slot Filling).
3. 모든 정보가 채워지면 학습 응원을 해줘.
답변:"""

prompt = PromptTemplate.from_template(template)

# 4. 데이터 흐름 정의
def get_history_string(_):
    messages = history_db.messages
    return "\n".join([f"{'Human' if i%2==0 else 'AI'}: {m.content}" for i, m in enumerate(messages)])

lcel_chain = (
    {
        "question": RunnablePassthrough(),
        "history": RunnableLambda(get_history_string),
        "name": lambda _: dialog_state["slots"]["name"] or "미파악",
        "location": lambda _: dialog_state["slots"]["location"] or "미파악",
        "topic": lambda _: dialog_state["slots"]["topic"] or "미파악"
    }
    | prompt
    | llm
    | StrOutputParser()
)

# 5. 실행 및 상태 업데이트 (Slot Filling 로직)
def chat(user_input):
    response = lcel_chain.invoke(user_input)

    # [간이 NLU] 발화에서 슬롯 추출 및 상태 업데이트 (실무에선 별도 모델 사용)
    if "제미니" in user_input: dialog_state["slots"]["name"] = "제미니"
    if "서울" in user_input: dialog_state["slots"]["location"] = "서울"
    if "랭체인" in user_input: dialog_state["slots"]["topic"] = "랭체인"

    history_db.add_user_message(user_input)
    history_db.add_ai_message(response)
    return response

print("08-1 지능형 대화관리가 반영된 AI 에이전트가 로드되었습니다.")

08-1 지능형 대화관리가 반영된 AI 에이전트가 로드되었습니다.


3. 실습 테스트 (Slot Filling 확인)  
- 단순한 대화가 아니라, 에이전트가 **부족한 정보(Slot)**를 인지하고 주도적으로 질문하는지 확인합니다.   

In [ ]:
# --- [08-1 테스트 시나리오] ---

# 1. 일부 정보만 제공 (Intent: 자기소개 / Slot: 이름 추출)
print("Q1: 안녕! 내 이름은 제미니야.")
print(f"AI: {chat('안녕! 내 이름은 제미니야.')}\n")

# 2. 에이전트의 주도권 확인 (비어있는 Slot인 거주지와 주제를 묻는지 확인)
print("Q2: 지금 서울에서 공부 중이야.")
print(f"AI: {chat('지금 서울에서 공부 중이야.')}\n")

# 3. 모든 Slot이 채워졌을 때의 반응 확인
print("Q3: 나는 요즘 랭체인을 배우고 있어.")
print(f"AI: {chat('나는 요즘 랭체인을 배우고 있어.')}\n")

# 4. 현재 최종 Dialog State 확인
print("📊 최종 Dialog State (슬롯 현황):", dialog_state["slots"])

Q1: 안녕! 내 이름은 제미니야.
AI: 안녕하세요, **제미니님!** 이름을 기억해 두겠습니다. 만나서 반갑습니다.

제미니님의 학습 계획을 효과적으로 지원하기 위해 몇 가지 정보가 더 필요합니다.

혹시 현재 **거주하고 계신 지역**은 어디신가요?

Q2: 지금 서울에서 공부 중이야.
AI: **서울**에서 열심히 공부하고 계시는군요! 기억해 두겠습니다.

이제 제미니님께서 어떤 학습 계획을 세우시는지 파악하기 위해 마지막 정보가 필요합니다.

혹시 **구체적으로 어떤 주제**에 대해 학습을 진행하고 싶으신가요?

Q3: 나는 요즘 랭체인을 배우고 있어.
AI: 랭체인(LangChain)에 대해 학습하고 계시는군요! 요즘 가장 주목받는 주제 중 하나이며, 복잡한 LLM 애플리케이션 개발에 필수적입니다. 기억해 두겠습니다.

**[최종 수집된 정보]**
*   **이름:** 제미니
*   **거주지:** 서울
*   **학습주제:** 랭체인

제미니님께서 서울에서 랭체인 학습을 시작하실 모든 준비가 완료되었습니다. 새로운 도전을 응원합니다! 목표하신 바를 이루실 수 있도록 제가 최선을 다해 돕겠습니다. 힘내세요!

📊 최종 Dialog State (슬롯 현황): {'name': '제미니', 'location': '서울', 'topic': '랭체인'}


#### 보강 포인트

- Stateful 관리: 기존 코드가 단순히 로그를 쌓았다면, 보강된 코드는 dialog_state 사전을 통해 어떤 데이터가 누락되었는지를 명시적으로 관리합니다.  
- Slot Filling 실습: 사용자가 정보를 다 주지 않았을 때, 에이전트가 프롬프트 내의 Dialog State 정보를 보고 스스로 미기입 항목을 질문하게 유도했습니다.  
- 목표 지향성: 대화의 끝이 '모든 슬롯을 채우는 것'으로 설정되어 08-1강의 학습 목표를 실습에 반영되었습니다.  